In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'Ready in: {os.getcwd()}')


In [ ]:
import numpy as np, pandas as pd, json, time
from pathlib import Path
from datetime import datetime
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
DS_LABEL = {'nsl_kdd_v2': 'NSL-KDD', 'unsw_nb15_v2': 'UNSW-NB15', 'cic_ids2017_v2': 'CIC-IDS2017'}
MODELS = [f'{a}_{v}' for v in ['5class_cw', '5class_smote'] for a in ['rf', 'xgb', 'dnn']]
CLASS_NAMES_5 = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
K = 5
EPS = 1e-12
ECE_N_BINS = 10
PLATT_THRESHOLD = 30          # 03e rule: Platt below this many fitting samples, isotonic otherwise
HOLDOUT_FRAC, RARE_CLASS_THRESHOLD = 0.20, 25   # 07e strict split
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42
RISK_TARGETS = [0.02, 0.05, 0.10]
COVERAGE_GRID = np.round(np.arange(0.50, 1.0001, 0.05), 2)
CALIBRATORS = ['raw', 'temperature', 'perclass_isotonic_renorm', 'toplabel_isotonic', 'cctl', 'cctl_shrunk', 'dirichlet']
KAPPA_GRID = [0, 25, 50, 100, 200, 400, 800, 1600]   # shrinkage strength grid; 0 = plain CCTL
TIE_BREAK_WITH_RAW = True   # isotonic maps produce ties; break them by the raw confidence (1e-6 scale) for ranking metrics
DECISION_PRESERVING = {'raw': True, 'temperature': True, 'perclass_isotonic_renorm': False, 'toplabel_isotonic': True, 'cctl': True, 'cctl_shrunk': True, 'dirichlet': False}

TABLES = Path(REPO) / 'results' / 'tables'
FIGS = Path(REPO) / 'results' / 'figures'
PREFIX = 'calib_methods_v2'

def find_proba_file(dataset, model_name, split):
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / f'{model_name}_{split}_proba.npy'
        if p.exists():
            return p
    raise FileNotFoundError(f'{dataset}/{model_name}_{split}_proba.npy')

def strict_split(y_calib):
    # Identical to 07e / notebook 12.
    n = len(y_calib); rng = np.random.RandomState(SEED)
    counts = Counter(y_calib.tolist())
    rare = [c for c, k in counts.items() if k < RARE_CLASS_THRESHOLD]
    common = [c for c, k in counts.items() if k >= RARE_CLASS_THRESHOLD]
    mask = np.zeros(n, dtype=bool)
    for c in rare:
        mask[y_calib == c] = True
    for c in common:
        idx = np.where(y_calib == c)[0]; rng.shuffle(idx)
        mask[idx[:int(round(len(idx) * (1 - HOLDOUT_FRAC)))]] = True
    return np.where(mask)[0], np.where(~mask)[0]
print('ready')


In [ ]:
def fit_binary_map(x, y):
    # Hybrid rule from 03e: isotonic when at least PLATT_THRESHOLD fitting samples, Platt otherwise;
    # falls back to identity when a stratum has no samples or a single label value.
    if len(x) < 2 or len(np.unique(y)) < 2:
        return lambda p: p
    if len(x) >= PLATT_THRESHOLD:
        iso = IsotonicRegression(out_of_bounds='clip').fit(x, y)
        return lambda p: iso.predict(p)
    lr = LogisticRegression(C=1e10, solver='lbfgs').fit(x.reshape(-1, 1), y)
    return lambda p: lr.predict_proba(p.reshape(-1, 1))[:, 1]

def redistribute(P, conf_new):
    # Full probability vector for a decision-preserving calibrator: the predicted class carries conf_new and the
    # residual mass is spread over the other classes in proportion to their raw probabilities, water-filled so
    # that no other class exceeds conf_new. When conf_new < 1/K no such vector exists; the decision is still the
    # raw argmax and the trust score is still conf_new (both are returned separately by transform()).
    n = len(P); pred = P.argmax(1)
    others = P.copy(); others[np.arange(n), pred] = 0.0
    w = others / np.maximum(others.sum(1, keepdims=True), EPS)
    out = w * (1 - conf_new)[:, None]
    cap = np.maximum(conf_new - 1e-9, 0)[:, None]
    for _ in range(K):
        over = out > cap
        if not over.any():
            break
        excess = np.clip(out - cap, 0, None).sum(1)
        out = np.where(over, cap, out)
        free = (~over) & (others > 0)
        fw = np.where(free, w, 0.0); fs = fw.sum(1)
        out = out + np.where(fs[:, None] > 0, fw / np.maximum(fs, EPS)[:, None] * excess[:, None], 0.0)
    out[np.arange(n), pred] = conf_new
    return out / np.maximum(out.sum(1, keepdims=True), EPS)

class Raw:
    def fit(self, P, y): return self
    def transform(self, P): return P, P.argmax(1), P.max(1), P.max(1)

class Temperature:
    # Single temperature on log-probabilities, fitted by NLL; decision-preserving.
    def fit(self, P, y):
        L = np.log(P + EPS)
        def nll(logT):
            Z = L / np.exp(logT); Z = Z - Z.max(1, keepdims=True)
            logp = Z - np.log(np.exp(Z).sum(1, keepdims=True))
            return -logp[np.arange(len(y)), y].mean()
        self.T = float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method='bounded').x)); return self
    def transform(self, P):
        Z = np.log(P + EPS) / self.T; Z = Z - Z.max(1, keepdims=True); E = np.exp(Z); Q = E / E.sum(1, keepdims=True)
        c = Q[np.arange(len(P)), P.argmax(1)]; return Q, P.argmax(1), c, c

class PerClassIsotonicRenorm:
    # The 03e recipe: one-vs-rest map per class, then renormalise. Not decision-preserving.
    def fit(self, P, y):
        self.maps = [fit_binary_map(P[:, c], (y == c).astype(int)) for c in range(K)]; return self
    def transform(self, P):
        Q = np.column_stack([self.maps[c](P[:, c]) for c in range(K)])
        s = Q.sum(1, keepdims=True); Q = np.where(s > 0, Q / np.maximum(s, EPS), P)
        return Q, Q.argmax(1), Q.max(1), Q.max(1)

class TopLabelIsotonic:
    # One map from top-label confidence to correctness, pooled over classes; argmax kept.
    def fit(self, P, y):
        conf = P.max(1); self.map = fit_binary_map(conf, (P.argmax(1) == y).astype(int)); return self
    def transform(self, P):
        conf = np.clip(self.map(P.max(1)), 0, 1); return redistribute(P, conf), P.argmax(1), conf, tie_break(conf, P.max(1))

class CCTL:
    # Class-conditional top-label calibration: one map per predicted class from top-label confidence to
    # correctness, argmax kept, residual mass redistributed. Fallback to the pooled map for classes with
    # too few fitting samples.
    def fit(self, P, y):
        conf, pred, corr = P.max(1), P.argmax(1), (P.argmax(1) == y).astype(int)
        self.pooled = fit_binary_map(conf, corr)
        self.maps, self.n_fit = {}, {}
        for c in range(K):
            m = pred == c; self.n_fit[c] = int(m.sum())
            self.maps[c] = fit_binary_map(conf[m], corr[m]) if m.sum() >= 2 and len(np.unique(corr[m])) == 2 else self.pooled
        return self
    def transform(self, P):
        conf, pred = P.max(1), P.argmax(1); out = np.empty(len(P))
        for c in range(K):
            m = pred == c
            if m.any():
                out[m] = self.maps[c](conf[m])
        out = np.clip(out, 0, 1); return redistribute(P, out), pred, out, tie_break(out, conf)

class Dirichlet:
    # Multinomial logistic regression on log-probabilities (Dirichlet calibration, L2 regularised). Not decision-preserving.
    def fit(self, P, y):
        self.lr = LogisticRegression(C=1.0, max_iter=2000).fit(np.log(P + EPS), y); self.classes = self.lr.classes_; return self
    def transform(self, P):
        Q = np.zeros((len(P), K)); Q[:, self.classes] = self.lr.predict_proba(np.log(P + EPS)); return Q, Q.argmax(1), Q.max(1), Q.max(1)

def tie_break(conf_cal, conf_raw):
    # Isotonic maps collapse many confidences onto identical values. For ranking, order within a tie group by the raw
    # confidence at a scale (1e-6) that cannot move any bin assignment or change the calibrated value materially.
    return conf_cal * (1 - 1e-6) + 1e-6 * conf_raw if TIE_BREAK_WITH_RAW else conf_cal

class CCTLShrunk:
    # Class-conditional top-label calibration with partial pooling: each class map is blended with the pooled map,
    # weight n_c / (n_c + kappa); kappa chosen on the holdout by binary log-loss of the calibrated confidence against
    # correctness. kappa = 0 recovers CCTL, kappa -> infinity recovers the pooled map.
    def __init__(self, kappa_grid=KAPPA_GRID):
        self.kappa_grid = kappa_grid
    def _blend(self, P, kappa):
        conf, pred = P.max(1), P.argmax(1); out = np.empty(len(P))
        for c in range(K):
            m = pred == c
            if m.any():
                w = self.n_fit[c] / (self.n_fit[c] + kappa) if (self.n_fit[c] + kappa) > 0 else 0.0
                out[m] = w * self.maps[c](conf[m]) + (1 - w) * self.pooled(conf[m])
        return np.clip(out, 0, 1)
    def fit(self, P, y, P_hold=None, y_hold=None):
        conf, pred, corr = P.max(1), P.argmax(1), (P.argmax(1) == y).astype(int)
        self.pooled = fit_binary_map(conf, corr); self.maps, self.n_fit = {}, {}
        for c in range(K):
            m = pred == c; self.n_fit[c] = int(m.sum())
            self.maps[c] = fit_binary_map(conf[m], corr[m]) if m.sum() >= 2 and len(np.unique(corr[m])) == 2 else self.pooled
        if P_hold is None:
            self.kappa = 100; return self
        corr_h = (P_hold.argmax(1) == y_hold).astype(float); best = None
        for kappa in self.kappa_grid:
            q = np.clip(self._blend(P_hold, kappa), 1e-6, 1 - 1e-6)
            ll = -np.mean(corr_h * np.log(q) + (1 - corr_h) * np.log(1 - q))
            if best is None or ll < best[0]:
                best = (ll, kappa)
        self.kappa = best[1]; self.holdout_logloss = best[0]; return self
    def transform(self, P):
        conf = self._blend(P, self.kappa); return redistribute(P, conf), P.argmax(1), conf, tie_break(conf, P.max(1))

FACTORY = {'raw': Raw, 'temperature': Temperature, 'perclass_isotonic_renorm': PerClassIsotonicRenorm,
           'toplabel_isotonic': TopLabelIsotonic, 'cctl': CCTL, 'cctl_shrunk': CCTLShrunk, 'dirichlet': Dirichlet}
print('calibrators ready')


In [ ]:
def bin_ids(p, n_bins=ECE_N_BINS):
    return np.clip((p * n_bins).astype(int), 0, n_bins - 1)

def ece_binned(p, lab, bins, n_bins=ECE_N_BINS):
    cnt = np.bincount(bins, minlength=n_bins).astype(float)
    sp = np.bincount(bins, weights=p, minlength=n_bins); sy = np.bincount(bins, weights=lab, minlength=n_bins)
    return float(np.abs(sp - sy)[cnt > 0].sum() / len(p))

def auroc(score, y):
    # Mann-Whitney with tie handling
    from scipy.stats import rankdata
    r = rankdata(score); n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if 0 < n1 < len(y) else float('nan')

def stratified_boot_indices(y, B, seed):
    rng = np.random.RandomState(seed); groups = [np.where(y == c)[0] for c in np.unique(y)]
    out = np.empty((B, len(y)), dtype=np.int64)
    for b in range(B):
        pos = 0
        for g in groups:
            out[b, pos:pos + len(g)] = rng.choice(g, size=len(g), replace=True); pos += len(g)
    return out

def point_metrics(P, pred, conf, score, y):
    corr = (pred == y).astype(float)
    Y = np.stack([(y == c).astype(float) for c in range(K)], 1)
    conf_mat = np.bincount(y * K + pred, minlength=K * K).reshape(K, K).astype(float)
    tp = np.diag(conf_mat); fp = conf_mat.sum(0) - tp; fn = conf_mat.sum(1) - tp
    with np.errstate(invalid='ignore', divide='ignore'):
        f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), 0.0)
        recall = np.where(tp + fn > 0, tp / (tp + fn), np.nan)
    return {'accuracy': float(corr.mean()), 'macro_f1': float(f1.mean()),
            'ece_top': ece_binned(conf, corr, bin_ids(conf)),
            'ece_macro': float(np.mean([ece_binned(P[:, c], Y[:, c], bin_ids(P[:, c])) for c in range(K)])),
            'brier_macro': float(((P - Y) ** 2).mean()),
            'nll': float(-np.log(P[np.arange(len(y)), y] + EPS).mean()),
            'auroc_conf_vs_correct': auroc(score, corr),
            **{f'recall_{CLASS_NAMES_5[c]}': float(recall[c]) for c in range(K)}}

def boot_metrics(P, pred, conf, score, y, idx):
    # ece_top and brier on the calibrated confidence, auroc on the ranking score, over stratified resamples
    corr = (pred == y).astype(float); bins = bin_ids(conf)
    Y = np.stack([(y == c).astype(float) for c in range(K)], 1)
    from scipy.stats import rankdata
    B = idx.shape[0]; e, br, au = np.empty(B), np.empty(B), np.empty(B)
    for b in range(B):
        s = idx[b]
        e[b] = ece_binned(conf[s], corr[s], bins[s]); br[b] = float(((P[s] - Y[s]) ** 2).mean()); au[b] = auroc(score[s], corr[s])
    return e, br, au

def risk_coverage(conf, corr, grid=COVERAGE_GRID):
    order = np.argsort(-conf); err = 1 - corr[order]; n = len(conf)
    cum = np.cumsum(err) / np.arange(1, n + 1)
    risks = [float(cum[max(int(np.ceil(c * n)) - 1, 0)]) for c in grid]
    aurc = float(cum.mean())
    return risks, aurc

def gate_thresholds(conf, pred, corr, target, per_class=True):
    # Smallest confidence threshold on the fitting surface such that selective risk among accepted samples <= target.
    def thr(c_, k_):
        order = np.argsort(-c_); e = 1 - k_[order]; cum = np.cumsum(e) / np.arange(1, len(e) + 1)
        ok = np.where(cum <= target)[0]
        if len(ok) == 0:
            return float('inf')
        j = ok.max(); return float(c_[order][j])
    if not per_class:
        return {c: thr(conf, corr) for c in range(K)}
    out = {}
    for c in range(K):
        m = pred == c
        out[c] = thr(conf[m], corr[m]) if m.sum() >= 30 else thr(conf, corr)
    return out

def apply_gate(conf, pred, corr, thresholds):
    t = np.array([thresholds[int(p)] for p in pred]); acc = conf >= t
    cov = float(acc.mean()); risk = float((1 - corr[acc]).mean()) if acc.any() else float('nan')
    per_class = {}
    for c in range(K):
        m = pred == c
        per_class[c] = (float(acc[m].mean()) if m.any() else float('nan'), float((1 - corr[m & acc]).mean()) if (m & acc).any() else float('nan'))
    return cov, risk, per_class
print('metrics ready')


In [ ]:
t0 = time.time()
metric_rows, rc_rows, gate_rows, flip_rows, temp_rows = [], [], [], [], []
curves = {}
for ds in DATASETS:
    proc = Path(REPO) / 'data' / 'processed' / ds
    y_cal = np.load(proc / 'y_calib_5class.npy'); y_te = np.load(proc / 'y_test_5class.npy')
    cached = Path(REPO) / 'calibrators' / ds / 'X_calib_strict_indices.npy'
    if cached.exists():
        strict_idx = np.load(cached); hold_idx = np.setdiff1d(np.arange(len(y_cal)), strict_idx)
    else:
        strict_idx, hold_idx = strict_split(y_cal)
    idx_boot = stratified_boot_indices(y_te, N_BOOTSTRAP, BOOTSTRAP_SEED)
    print(f'\n{ds}: strict={len(strict_idx)} holdout={len(hold_idx)} test={len(y_te)}')
    for m in MODELS:
        P_cal = np.load(find_proba_file(ds, m, 'calib')).astype(np.float64); P_te = np.load(find_proba_file(ds, m, 'test')).astype(np.float64)
        P_cal /= np.maximum(P_cal.sum(1, keepdims=True), EPS); P_te /= np.maximum(P_te.sum(1, keepdims=True), EPS)
        Ps, ys, Ph, yh = P_cal[strict_idx], y_cal[strict_idx], P_cal[hold_idx], y_cal[hold_idx]
        raw_pred = P_te.argmax(1)
        base = None
        for name in CALIBRATORS:
            cal = FACTORY[name]().fit(Ps, ys, Ph, yh) if name == 'cctl_shrunk' else FACTORY[name]().fit(Ps, ys)
            Q_te, pred_te, conf_te, score_te = cal.transform(P_te); Q_h, pred_h, conf_h, score_h = cal.transform(Ph)
            pm = point_metrics(Q_te, pred_te, conf_te, score_te, y_te)
            e, br, au = boot_metrics(Q_te, pred_te, conf_te, score_te, y_te, idx_boot)
            if name == 'raw':
                base = (e, br, au)
            row = {'dataset': ds, 'model': m, 'calibrator': name, 'decision_preserving_by_construction': DECISION_PRESERVING[name],
                   'pct_decisions_changed_vs_raw': 100 * float((pred_te != raw_pred).mean()), **pm,
                   'ece_top_lo': float(np.percentile(e, 2.5)), 'ece_top_hi': float(np.percentile(e, 97.5)),
                   'brier_macro_lo': float(np.percentile(br, 2.5)), 'brier_macro_hi': float(np.percentile(br, 97.5)),
                   'auroc_lo': float(np.percentile(au, 2.5)), 'auroc_hi': float(np.percentile(au, 97.5)),
                   'ece_top_delta_vs_raw_lo': float(np.percentile(e - base[0], 2.5)), 'ece_top_delta_vs_raw_hi': float(np.percentile(e - base[0], 97.5)),
                   'brier_delta_vs_raw_lo': float(np.percentile(br - base[1], 2.5)), 'brier_delta_vs_raw_hi': float(np.percentile(br - base[1], 97.5)),
                   'auroc_delta_vs_raw_lo': float(np.percentile(au - base[2], 2.5)), 'auroc_delta_vs_raw_hi': float(np.percentile(au - base[2], 97.5))}
            if name == 'temperature':
                row['temperature_T'] = cal.T
            if name == 'cctl_shrunk':
                row['kappa'] = cal.kappa; row['holdout_logloss'] = cal.holdout_logloss
            for c in range(K):
                mc = pred_te == c
                row[f'stratum_ece_{CLASS_NAMES_5[c]}'] = ece_binned(conf_te[mc], (pred_te[mc] == y_te[mc]).astype(float), bin_ids(conf_te[mc])) if mc.sum() >= 30 else float('nan')
                row[f'stratum_n_{CLASS_NAMES_5[c]}'] = int(mc.sum())
            metric_rows.append(row)
            corr_te = (pred_te == y_te).astype(float)
            risks, aurc = risk_coverage(score_te, corr_te)
            curves[(ds, m, name)] = risks
            rc_rows.append({'dataset': ds, 'model': m, 'calibrator': name, 'aurc': aurc, **{f'risk_at_cov_{c:.2f}': r for c, r in zip(COVERAGE_GRID, risks)}})
            corr_h = (pred_h == yh).astype(float)
            for target in RISK_TARGETS:
                for pc in [True, False]:
                    th = gate_thresholds(score_h, pred_h, corr_h, target, per_class=pc)
                    cov, risk, per_class = apply_gate(score_te, pred_te, corr_te, th)
                    gate_rows.append({'dataset': ds, 'model': m, 'calibrator': name, 'target_risk': target, 'per_class_thresholds': pc,
                                      'coverage': cov, 'achieved_risk': risk, 'risk_met': bool(risk <= target * 1.1) if np.isfinite(risk) else False,
                                      **{f'thr_{CLASS_NAMES_5[c]}': th[c] for c in range(K)},
                                      **{f'cov_{CLASS_NAMES_5[c]}': per_class[c][0] for c in range(K)},
                                      **{f'risk_{CLASS_NAMES_5[c]}': per_class[c][1] for c in range(K)}})
            print(f'  {m:18s} {name:26s} acc={pm["accuracy"]:.3f} F1={pm["macro_f1"]:.3f} ECEtop={pm["ece_top"]:.4f} Brier={pm["brier_macro"]:.4f} '
                  f'AUROC={pm["auroc_conf_vs_correct"]:.3f} changed={row["pct_decisions_changed_vs_raw"]:5.1f}% DoS_rec={pm["recall_DoS"]:.2f} R2L_rec={pm["recall_R2L"]:.2f} AURC={aurc:.4f}' + (f' kappa={cal.kappa}' if name == 'cctl_shrunk' else ''))
df_m = pd.DataFrame(metric_rows); df_m.to_csv(TABLES / f'{PREFIX}_metrics.csv', index=False)
df_rc = pd.DataFrame(rc_rows); df_rc.to_csv(TABLES / f'{PREFIX}_risk_coverage.csv', index=False)
df_g = pd.DataFrame(gate_rows); df_g.to_csv(TABLES / f'{PREFIX}_gate.csv', index=False)
print(f'\n{(time.time() - t0) / 60:.1f} min; {len(df_m)} metric rows, {len(df_rc)} risk-coverage rows, {len(df_g)} gate rows')


In [ ]:
pd.set_option('display.width', 260)
def verdict(lo, hi, lower_is_better=True):
    if lower_is_better:
        return 'better' if hi < 0 else ('worse' if lo > 0 else 'no difference')
    return 'better' if lo > 0 else ('worse' if hi < 0 else 'no difference')
df_m['ece_top_verdict_vs_raw'] = [verdict(a, b) for a, b in zip(df_m.ece_top_delta_vs_raw_lo, df_m.ece_top_delta_vs_raw_hi)]
df_m['brier_verdict_vs_raw'] = [verdict(a, b) for a, b in zip(df_m.brier_delta_vs_raw_lo, df_m.brier_delta_vs_raw_hi)]
df_m['auroc_verdict_vs_raw'] = [verdict(a, b, False) for a, b in zip(df_m.auroc_delta_vs_raw_lo, df_m.auroc_delta_vs_raw_hi)]
df_m.to_csv(TABLES / f'{PREFIX}_metrics.csv', index=False)

summ = df_m.groupby(['dataset', 'calibrator']).agg(
    ece_top=('ece_top', 'mean'), brier_macro=('brier_macro', 'mean'), nll=('nll', 'mean'), auroc=('auroc_conf_vs_correct', 'mean'),
    accuracy=('accuracy', 'mean'), macro_f1=('macro_f1', 'mean'), pct_changed=('pct_decisions_changed_vs_raw', 'mean'),
    recall_DoS=('recall_DoS', 'mean'), recall_R2L=('recall_R2L', 'mean'), recall_U2R=('recall_U2R', 'mean'),
    ece_better=('ece_top_verdict_vs_raw', lambda s: int((s == 'better').sum())), ece_worse=('ece_top_verdict_vs_raw', lambda s: int((s == 'worse').sum())),
    brier_better=('brier_verdict_vs_raw', lambda s: int((s == 'better').sum())), brier_worse=('brier_verdict_vs_raw', lambda s: int((s == 'worse').sum())),
    auroc_better=('auroc_verdict_vs_raw', lambda s: int((s == 'better').sum())), auroc_worse=('auroc_verdict_vs_raw', lambda s: int((s == 'worse').sum()))).reset_index()
summ['calibrator'] = pd.Categorical(summ.calibrator, CALIBRATORS); summ = summ.sort_values(['dataset', 'calibrator'])
summ.to_csv(TABLES / f'{PREFIX}_summary.csv', index=False)
print(summ.round(4).to_string(index=False))

gs = df_g.groupby(['dataset', 'calibrator', 'target_risk', 'per_class_thresholds']).agg(
    coverage=('coverage', 'mean'), achieved_risk=('achieved_risk', 'mean'), models_meeting_target=('risk_met', 'sum')).reset_index()
gs.to_csv(TABLES / f'{PREFIX}_gate_summary.csv', index=False)
print('\ngate (target risk -> mean achieved risk and coverage over the six models; thresholds fitted on the held-out calibration slice)')
print(gs[gs.calibrator.isin(['raw', 'perclass_isotonic_renorm', 'toplabel_isotonic', 'cctl', 'cctl_shrunk'])].round(3).to_string(index=False))

plt.rcParams.update({'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.4), sharey=False)
style = {'raw': ('#888888', ':'), 'temperature': ('#4477AA', '--'), 'perclass_isotonic_renorm': ('#CC3311', '-.'), 'cctl': ('#000000', '--'), 'cctl_shrunk': ('#000000', '-'), 'toplabel_isotonic': ('#009988', '-'), 'dirichlet': ('#EE7733', '--')}
for ax, ds in zip(axes, DATASETS):
    for name in CALIBRATORS:
        R = np.mean([curves[(ds, m, name)] for m in MODELS], axis=0)
        ax.plot(COVERAGE_GRID, R, color=style[name][0], ls=style[name][1], lw=1.1, label=name.replace('_', ' '))
    ax.set_title(DS_LABEL[ds]); ax.set_xlabel('Coverage'); ax.grid(lw=0.3, alpha=0.5)
axes[0].set_ylabel('Selective risk (mean over models)'); axes[2].legend(frameon=False, fontsize=6)
fig.tight_layout(); fig.savefig(FIGS / f'{PREFIX}_risk_coverage.png', dpi=300, bbox_inches='tight'); fig.savefig(FIGS / f'{PREFIX}_risk_coverage.pdf', bbox_inches='tight')
with open(TABLES / f'{PREFIX}_summary.json', 'w') as f:
    json.dump({'timestamp': datetime.now().isoformat(), 'notebook': '16b_cctl_shrinkage.ipynb', 'calibrators': CALIBRATORS, 'kappa_grid': KAPPA_GRID, 'tie_break_with_raw': TIE_BREAK_WITH_RAW,
               'decision_preserving': DECISION_PRESERVING, 'fit_surface': 'strict 80% slice of the calibration partition (07e split)',
               'gate_surface': '20% holdout of the calibration partition', 'n_bootstrap': N_BOOTSTRAP, 'risk_targets': RISK_TARGETS}, f, indent=2)
print('saved figure and summary')


In [ ]:
os.chdir(REPO)
!git config user.name "Md Anas Biswas"
!git config user.email "anasbiswas@gmail.com"
import nbformat as _nbf
_nb_path = Path(REPO) / 'notebooks' / '16b_cctl_shrinkage.ipynb'
if _nb_path.exists():
    _nb = _nbf.read(_nb_path, 4)
    for _c in _nb.cells:
        if _c.cell_type == 'code':
            _c.outputs, _c.execution_count = [], None
    _nbf.write(_nb, _nb_path)
    print(f'outputs stripped: {_nb_path.relative_to(REPO)}')
else:
    raise FileNotFoundError(f'{_nb_path} not found: save this notebook under notebooks/ before committing')
!git add notebooks/16b_cctl_shrinkage.ipynb results/tables/calib_methods_v2_*.csv results/tables/calib_methods_v2_summary.json results/figures/calib_methods_v2_risk_coverage.png results/figures/calib_methods_v2_risk_coverage.pdf
!git status --short | head -20
!git commit -m "Notebook 16b: CCTL with partial pooling (kappa chosen on the holdout by top-label log-loss), raw-confidence tie-breaking for isotonic maps, per-stratum ECE; full rerun of the six calibrators plus the shrunk variant as calib_methods_v2_*"
!git push origin main
!git log --oneline -2
